# Notebook 24 — Race-Time Database Extension

## Purpose

Study 01 requires direct database access to the canonical race-time information already governed by Notebook 11.

The accepted Inside Rails Version 1 database must remain immutable. This investigation therefore asks whether the existing governed race-time output can support a separately validated database extension and, if so, what the smallest safe extension should contain.

The first bounded question is:

> Does `canonical_race_times.csv` attach exactly one-to-one to all 189,043 accepted `core_source_race_occurrence` rows using the authorised Source Version 1 race key `date + course + off`?

No schema, migration or database write is authorised until that relationship has been proved.

## Existing governed temporal output

The existing Notebook 11 output contains, where resolved:

- `advertised_start_uk`;
- `advertised_start_utc`;
- `advertised_start_course_local`;
- `selected_branch`;
- `decision_method`;
- `decision_confidence`;
- `temporal_resolution_status`.

Unresolved races must remain unresolved. The raw source `date`, `course` and `off` values must remain unchanged.

In [1]:
from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# Resolve the repository root without assuming that Jupyter was launched from
# either the repository root or the notebooks directory.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Use only the documented accepted Database v1 release and the already-governed
# Notebook 11 race-time output. Neither input is modified by this diagnostic.
DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v1.sqlite3"
)
RACE_TIMES = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "race_times"
    / "canonical_race_times.csv"
)

# Fail closed if either governed input is unavailable rather than silently
# substituting a candidate database, raw source file or reconstructed output.
assert DATABASE.is_file(), f"Accepted Database v1 not found: {DATABASE}"
assert RACE_TIMES.is_file(), f"Canonical race-time output not found: {RACE_TIMES}"

# Read only the authorised raw race key from the accepted database. Casting to
# text gives us an exact comparable representation without altering stored data.
with connect_read_only(DATABASE) as connection:
    connection.execute("PRAGMA query_only = ON")
    connection.execute("PRAGMA foreign_keys = ON")
    connection.execute("PRAGMA trusted_schema = OFF")

    database_races = pd.read_sql_query(
        """
        SELECT
            CAST(raw_date AS TEXT) AS date,
            CAST(raw_course AS TEXT) AS course,
            CAST(raw_off AS TEXT) AS off
        FROM core_source_race_occurrence
        ORDER BY source_race_occurrence_id
        """,
        connection,
    )

# Load only the three governed race-key columns at this stage. We are testing
# attachment cardinality before considering which temporal fields belong in a
# database extension.
race_times = pd.read_csv(
    RACE_TIMES,
    usecols=["date", "course", "off"],
    dtype={"date": "string", "course": "string", "off": "string"},
)

# Each side must independently contain exactly one row per authorised race key.
# A duplicate on either side would make a one-to-one database attachment unsafe.
database_duplicate_keys = int(
    database_races.duplicated(["date", "course", "off"], keep=False).sum()
)
race_time_duplicate_keys = int(
    race_times.duplicated(["date", "course", "off"], keep=False).sum()
)

# An outer one-to-one merge proves both cardinality and coverage. `validate`
# deliberately raises if either input unexpectedly ceases to be unique.
reconciliation = database_races.merge(
    race_times,
    on=["date", "course", "off"],
    how="outer",
    indicator=True,
    validate="one_to_one",
)

reconciliation_counts = (
    reconciliation["_merge"]
    .value_counts()
    .reindex(["both", "left_only", "right_only"], fill_value=0)
)

print(f"Accepted database races: {len(database_races):,}")
print(f"Canonical race-time rows: {len(race_times):,}")
print(f"Database duplicate-key rows: {database_duplicate_keys:,}")
print(f"Race-time duplicate-key rows: {race_time_duplicate_keys:,}")
print()
print("Race-key reconciliation:")
print(reconciliation_counts.to_string())

Accepted database races: 189,043
Canonical race-time rows: 189,043
Database duplicate-key rows: 0
Race-time duplicate-key rows: 0

Race-key reconciliation:
_merge
both          189043
left_only          0
right_only         0


### Result — canonical race times attach one-to-one to the structural race core

The governed Notebook 11 race-time output reconciles exactly to the accepted database race population.

Both inputs contain 189,043 races. The authorised Source Version 1 race key `date + course + off` is unique on both sides, and an outer one-to-one reconciliation produced:

- 189,043 matched races;
- 0 database-only races;
- 0 race-time-only races.

This establishes that every governed race-time record has exactly one structural `core_source_race_occurrence` attachment point and that no database race would be omitted by the extension.

It does not yet establish the physical extension schema or which temporal fields can be stored as non-null values. The next question is therefore:

> Do the governed temporal fields, resolution statuses and null patterns satisfy the existing Notebook 11 persistence contract across all 189,043 races?

In [2]:
# Load the complete governed temporal payload at race grain. We keep the
# timestamp columns as strings here because this stage is checking the persisted
# representation that a database extension would actually ingest.
temporal = pd.read_csv(
    RACE_TIMES,
    dtype="string",
)

# These are the fields required by the existing Notebook 11 integration
# contract. Fail immediately if the persisted governed output has drifted.
required_columns = {
    "date",
    "course",
    "off",
    "iana_timezone",
    "candidate_a_uk_naive",
    "candidate_b_uk_naive",
    "candidate_a_utc",
    "candidate_b_utc",
    "candidate_a_course_local",
    "candidate_b_course_local",
    "advertised_start_uk",
    "advertised_start_utc",
    "advertised_start_course_local",
    "selected_branch",
    "decision_method",
    "decision_confidence",
    "temporal_resolution_status",
}

missing_columns = required_columns - set(temporal.columns)
assert not missing_columns, (
    "Canonical race-time output is missing governed fields: "
    f"{sorted(missing_columns)}"
)

# Confirm the persisted output still has the already-established complete race
# population before inspecting status-specific null behaviour.
assert len(temporal) == 189_043, (
    f"Unexpected canonical race-time population: {len(temporal):,}"
)

# Split only on the governed resolution status. We do not infer resolution from
# whether a timestamp happens to be populated.
status_counts = temporal["temporal_resolution_status"].value_counts(dropna=False)

resolved = temporal["temporal_resolution_status"].eq("resolved")
unresolved = temporal["temporal_resolution_status"].eq("unresolved")

# A resolved race must carry all three selected canonical representations.
resolved_missing_selected = temporal.loc[
    resolved,
    [
        "advertised_start_uk",
        "advertised_start_utc",
        "advertised_start_course_local",
    ],
].isna().any(axis=1).sum()

# An unresolved race must not contain a selected canonical timestamp or selected
# branch. Preserving nulls here is essential: the database must not turn an
# unresolved Notebook 11 decision into false precision.
unresolved_with_selected_timestamp = temporal.loc[
    unresolved,
    [
        "advertised_start_uk",
        "advertised_start_utc",
        "advertised_start_course_local",
    ],
].notna().any(axis=1).sum()

unresolved_with_selected_branch = (
    temporal.loc[unresolved, "selected_branch"].notna().sum()
)

# Notebook 11 deliberately preserves both temporal candidates for unresolved
# pre-boundary races. Verify that this evidence has survived persistence.
candidate_columns = [
    "candidate_a_uk_naive",
    "candidate_b_uk_naive",
    "candidate_a_utc",
    "candidate_b_utc",
    "candidate_a_course_local",
    "candidate_b_course_local",
]

unresolved_missing_candidate = temporal.loc[
    unresolved,
    candidate_columns,
].isna().any(axis=1).sum()

print("Resolution status:")
print(status_counts.to_string())

print()
print(f"Resolved rows missing a selected timestamp: {resolved_missing_selected:,}")
print(
    "Unresolved rows containing a selected timestamp: "
    f"{unresolved_with_selected_timestamp:,}"
)
print(
    "Unresolved rows containing a selected branch: "
    f"{unresolved_with_selected_branch:,}"
)
print(
    "Unresolved rows missing one or more preserved candidates: "
    f"{unresolved_missing_candidate:,}"
)

print()
print("Decision methods:")
print(temporal["decision_method"].value_counts(dropna=False).to_string())

Resolution status:
temporal_resolution_status
resolved      169465
unresolved     19578

Resolved rows missing a selected timestamp: 0
Unresolved rows containing a selected timestamp: 0
Unresolved rows containing a selected branch: 0
Unresolved rows missing one or more preserved candidates: 141

Decision methods:
decision_method
course_local_dead_of_night_rejection    111871
stable_post_boundary_course_profile      47242
unresolved                               19578
explicit_post_boundary_time              10352


### Result — selected temporal state reconciles; candidate nullability needs qualification

The canonical output reproduces the established Notebook 11 population exactly:

- 169,465 resolved races;
- 19,578 unresolved races;
- 111,871 resolved by course-local dead-of-night rejection;
- 47,242 resolved by stable post-boundary course profile;
- 10,352 explicit post-boundary races;
- 19,578 retained unresolved.

Every resolved race contains all three selected canonical timestamps. No unresolved race contains a selected timestamp or selected branch.

However, 141 unresolved races are missing at least one of the six preserved candidate representations.

This does not yet establish a defect. Candidate construction crosses historical civil-time rules, so the missing values may represent intentionally unconvertible daylight-saving edge cases rather than lost evidence.

The next bounded question is therefore:

> Exactly which candidate fields are missing in those 141 unresolved races, and do their null patterns correspond to governed DST ambiguity/nonexistence rather than unexplained missing data?

In [3]:
# Restrict investigation to the 141 unresolved rows that violated our overly
# strong assumption that all six candidate representations must be non-null.
candidate_exception_mask = (
    unresolved
    & temporal[candidate_columns].isna().any(axis=1)
)

candidate_exceptions = temporal.loc[
    candidate_exception_mask,
    [
        "date",
        "course",
        "off",
        "iana_timezone",
        *candidate_columns,
        "decision_method",
        "decision_confidence",
        "temporal_resolution_status",
    ],
].copy()

# Encode the exact null pattern rather than merely counting affected rows.
# This tells us whether the raw UK candidate exists but UTC/course-local
# conversion is unavailable, or whether candidate construction itself failed.
candidate_exceptions["missing_candidate_fields"] = candidate_exceptions[
    candidate_columns
].apply(
    lambda row: " | ".join(
        column for column in candidate_columns if pd.isna(row[column])
    ),
    axis=1,
)

null_pattern_counts = (
    candidate_exceptions["missing_candidate_fields"]
    .value_counts()
    .rename_axis("missing_candidate_fields")
    .reset_index(name="races")
)

# Count nulls independently by field as a second check on the pattern summary.
field_null_counts = (
    candidate_exceptions[candidate_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(f"Candidate-exception races: {len(candidate_exceptions):,}")

print("\nExact missing-field patterns:")
print(null_pattern_counts.to_string(index=False))

print("\nNull counts by candidate field:")
print(field_null_counts.to_string())

# Show a small bounded sample so we can inspect dates and clock values.
# We are not yet interpreting these as DST failures; the sample is evidence
# for deciding whether the next step should test that hypothesis.
print("\nSample exception rows:")
print(
    candidate_exceptions[
        [
            "date",
            "course",
            "off",
            "iana_timezone",
            "candidate_a_uk_naive",
            "candidate_b_uk_naive",
            "candidate_a_utc",
            "candidate_b_utc",
            "candidate_a_course_local",
            "candidate_b_course_local",
            "missing_candidate_fields",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

Candidate-exception races: 141

Exact missing-field patterns:
                  missing_candidate_fields  races
candidate_a_utc | candidate_a_course_local    138
candidate_b_utc | candidate_b_course_local      3

Null counts by candidate field:
candidate_a_course_local    138
candidate_a_utc             138
candidate_b_utc               3
candidate_b_course_local      3
candidate_b_uk_naive          0
candidate_a_uk_naive          0

Sample exception rows:
      date             course  off iana_timezone candidate_a_uk_naive candidate_b_uk_naive candidate_a_utc           candidate_b_utc candidate_a_course_local  candidate_b_course_local                   missing_candidate_fields
2015-03-29       Auteuil (FR) 1:00  Europe/Paris  2015-03-29T01:00:00  2015-03-29T13:00:00            <NA> 2015-03-29T12:00:00+00:00                     <NA> 2015-03-29T14:00:00+02:00 candidate_a_utc | candidate_a_course_local
2015-03-29       Auteuil (FR) 1:30  Europe/Paris  2015-03-29T01:30:00  2015-03-29T13:

### Result — candidate nulls have the structure expected at London DST edges

All 141 candidate exceptions preserve both reconstructed UK-naive candidate timestamps.

The missing values occur only in the UTC and corresponding course-local representations:

- 138 races lack candidate A UTC and course-local values;
- 3 races lack candidate B UTC and course-local values;
- no race lacks either UK-naive candidate.

This matches the Notebook 11 design in which an ambiguous or nonexistent `Europe/London` civil timestamp is preserved as a naive candidate but deliberately withheld from UTC conversion.

The next bounded question is:

> Are all 141 missing conversions explained exactly by Notebook 11's governed `ambiguous_dst_time` or `nonexistent_dst_time` classifications, with the opposite candidate remaining valid?

In [5]:
from inside_rails.race_times import classify_london_civil_time


# Classify only the 141 exceptional races. There is no reason to repeat the
# timezone operation across all 189,043 races when the bounded question concerns
# this already-isolated residue.
for branch in ("a", "b"):
    naive_column = f"candidate_{branch}_uk_naive"
    status_column = f"candidate_{branch}_london_status"

    candidate_exceptions[status_column] = candidate_exceptions[
        naive_column
    ].map(classify_london_civil_time)

# Identify which branch has its UTC conversion withheld. The earlier null-pattern
# result proved that exactly one branch is affected in each exceptional race.
candidate_exceptions["withheld_branch"] = pd.Series(
    pd.NA,
    index=candidate_exceptions.index,
    dtype="string",
)
candidate_exceptions.loc[
    candidate_exceptions["candidate_a_utc"].isna(),
    "withheld_branch",
] = "a"
candidate_exceptions.loc[
    candidate_exceptions["candidate_b_utc"].isna(),
    "withheld_branch",
] = "b"

# Pull the governed London classification corresponding to the withheld branch
# and, separately, the status of the opposite candidate.
candidate_exceptions["withheld_london_status"] = candidate_exceptions.apply(
    lambda row: row[f"candidate_{row['withheld_branch']}_london_status"],
    axis=1,
)

candidate_exceptions["opposite_london_status"] = candidate_exceptions.apply(
    lambda row: (
        row["candidate_b_london_status"]
        if row["withheld_branch"] == "a"
        else row["candidate_a_london_status"]
    ),
    axis=1,
)

# A withheld conversion is justified only by one of Notebook 11's two explicit
# DST-edge states. The opposite candidate should remain a valid London civil time.
unexpected_withheld_status = ~candidate_exceptions[
    "withheld_london_status"
].isin(
    ["ambiguous_dst_time", "nonexistent_dst_time"]
)

unexpected_opposite_status = ~candidate_exceptions[
    "opposite_london_status"
].eq("valid")

print("Withheld branch:")
print(candidate_exceptions["withheld_branch"].value_counts().to_string())

print("\nGoverned status of withheld candidate:")
print(
    candidate_exceptions["withheld_london_status"]
    .value_counts()
    .to_string()
)

print("\nGoverned status of opposite candidate:")
print(
    candidate_exceptions["opposite_london_status"]
    .value_counts()
    .to_string()
)

print()
print(
    "Withheld candidates with an unexpected London status: "
    f"{int(unexpected_withheld_status.sum()):,}"
)
print(
    "Opposite candidates not classified valid: "
    f"{int(unexpected_opposite_status.sum()):,}"
)

Withheld branch:
withheld_branch
a    138
b      3

Governed status of withheld candidate:
withheld_london_status
ambiguous_dst_time      96
nonexistent_dst_time    45

Governed status of opposite candidate:
opposite_london_status
valid    141

Withheld candidates with an unexpected London status: 0
Opposite candidates not classified valid: 0


### Result — all candidate conversion nulls are governed DST-edge states

The 141 candidate conversion exceptions are fully explained by the existing Notebook 11 temporal rules.

Of the withheld candidate branches:

- 96 are classified `ambiguous_dst_time`;
- 45 are classified `nonexistent_dst_time`;
- 0 have an unexpected London civil-time status.

For every one of the 141 races, the opposite 12-hour candidate is classified `valid`.

The missing UTC and course-local candidate values are therefore intentional governed nulls rather than missing data or persistence defects.

A database extension must preserve these nulls exactly. It must not manufacture UTC or course-local timestamps for ambiguous or nonexistent UK civil times.

The temporal evidence is now sufficiently reconciled to define the smallest database extension.

The next bounded question is:

> What exact field domains, nullability rules and timestamp representations must the race-time extension preserve?

In [6]:
# Profile only fields that could become part of the governed race-time database
# extension. This stage establishes physical constraints from the persisted
# evidence rather than inventing SQL nullability or enumerated domains.
extension_fields = [
    "iana_timezone",
    "candidate_a_uk_naive",
    "candidate_b_uk_naive",
    "candidate_a_utc",
    "candidate_b_utc",
    "candidate_a_course_local",
    "candidate_b_course_local",
    "advertised_start_uk",
    "advertised_start_utc",
    "advertised_start_course_local",
    "selected_branch",
    "decision_method",
    "decision_confidence",
    "temporal_resolution_status",
]

extension_profile = []

for column in extension_fields:
    series = temporal[column]

    extension_profile.append(
        {
            "field": column,
            "rows": len(series),
            "non_null": int(series.notna().sum()),
            "null": int(series.isna().sum()),
            "distinct_non_null": int(series.nunique(dropna=True)),
        }
    )

extension_profile = pd.DataFrame(extension_profile)

print("Field population profile:")
print(extension_profile.to_string(index=False))

# Enumerate the genuinely categorical governance fields. These domains should
# later become database CHECK constraints rather than unrestricted free text.
for column in [
    "selected_branch",
    "decision_method",
    "decision_confidence",
    "temporal_resolution_status",
]:
    print(f"\n{column}:")
    print(
        temporal[column]
        .value_counts(dropna=False)
        .to_string()
    )

# The course timezone is required to interpret course-local timestamps. Confirm
# that every race retains one governed IANA timezone and inspect its cardinality.
print("\nIANA timezone coverage:")
print(f"non-null: {int(temporal['iana_timezone'].notna().sum()):,}")
print(f"null: {int(temporal['iana_timezone'].isna().sum()):,}")
print(f"distinct: {temporal['iana_timezone'].nunique(dropna=True):,}")

# Inspect representative persisted timestamp strings rather than parsing or
# rewriting them. The extension should preserve the governed temporal meaning
# and use explicit ISO representations with timezone offsets where applicable.
for column in [
    "candidate_a_uk_naive",
    "candidate_a_utc",
    "candidate_a_course_local",
    "advertised_start_uk",
    "advertised_start_utc",
    "advertised_start_course_local",
]:
    examples = temporal.loc[temporal[column].notna(), column].head(3).tolist()

    print(f"\n{column} examples:")
    for value in examples:
        print(f"  {value}")

Field population profile:
                        field   rows  non_null  null  distinct_non_null
                iana_timezone 189043    189043     0                 51
         candidate_a_uk_naive 189043    178691 10352             169730
         candidate_b_uk_naive 189043    178691 10352             169730
              candidate_a_utc 189043    178553 10490             169603
              candidate_b_utc 189043    178688 10355             169727
     candidate_a_course_local 189043    178553 10490             177928
     candidate_b_course_local 189043    178688 10355             178063
          advertised_start_uk 189043    169465 19578             161565
         advertised_start_utc 189043    169465 19578             161565
advertised_start_course_local 189043    169465 19578             168875
              selected_branch 189043    169465 19578                  3
              decision_method 189043    189043     0                  4
          decision_confidence 189043  

### Result — field populations define the extension's basic nullability

The governed race-time output provides a complete temporal record for all 189,043 structural races.

Fields that are complete for every race are:

- `iana_timezone`;
- `decision_method`;
- `decision_confidence`;
- `temporal_resolution_status`.

The three selected canonical timestamps and `selected_branch` are populated for exactly the 169,465 resolved races and remain null for all 19,578 unresolved races.

The two UK-naive candidate timestamps are populated for all 178,691 pre-boundary races and absent for the 10,352 post-boundary races whose source time is already explicit 24-hour UK civil time.

Candidate UTC and course-local values require additional nullability because 141 pre-boundary candidate branches fall on governed ambiguous or nonexistent London DST civil times.

The persisted Notebook 11 output therefore already contains the evidence needed for the extension without inventing replacement timestamps.

Before writing SQL constraints, however, the exact permitted combinations of resolution status, selected branch, decision method and confidence must be established.

The next bounded question is:

> What exact governance-state combinations occur, and do pre-boundary and post-boundary races form clean mutually exclusive temporal states?

In [8]:
from inside_rails.race_times import FORMAT_BOUNDARY


# The SQL extension should admit only governance-state combinations that actually
# occur in the validated Notebook 11 output. Profile complete combinations rather
# than constructing independent CHECK domains that could permit impossible states.
governance_combinations = (
    temporal.groupby(
        [
            "temporal_resolution_status",
            "selected_branch",
            "decision_method",
            "decision_confidence",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="races")
    .sort_values("races", ascending=False)
)

print("Exact governance-state combinations:")
print(governance_combinations.to_string(index=False))

# Split the persisted rows on the governed 15 October 2025 format boundary.
# This tests whether the candidate/explicit-time states line up exactly with the
# historical source-format change already established by Notebook 11.
source_dates = pd.to_datetime(temporal["date"], errors="raise")
pre_boundary_mask = source_dates.lt(FORMAT_BOUNDARY)
post_boundary_mask = ~pre_boundary_mask

period_summary = pd.DataFrame(
    [
        {
            "period": "pre_boundary",
            "races": int(pre_boundary_mask.sum()),
            "with_candidate_a_naive": int(
                temporal.loc[pre_boundary_mask, "candidate_a_uk_naive"].notna().sum()
            ),
            "with_candidate_b_naive": int(
                temporal.loc[pre_boundary_mask, "candidate_b_uk_naive"].notna().sum()
            ),
            "explicit_24h_branch": int(
                temporal.loc[pre_boundary_mask, "selected_branch"]
                .eq("explicit_24h")
                .sum()
            ),
        },
        {
            "period": "post_boundary",
            "races": int(post_boundary_mask.sum()),
            "with_candidate_a_naive": int(
                temporal.loc[post_boundary_mask, "candidate_a_uk_naive"].notna().sum()
            ),
            "with_candidate_b_naive": int(
                temporal.loc[post_boundary_mask, "candidate_b_uk_naive"].notna().sum()
            ),
            "explicit_24h_branch": int(
                temporal.loc[post_boundary_mask, "selected_branch"]
                .eq("explicit_24h")
                .sum()
            ),
        },
    ]
)

print("\nPeriod/candidate structure:")
print(period_summary.to_string(index=False))

# Cross-check the selected canonical fields as one state. A row must contain
# either all three selected timestamps or none; partial selection would make a
# database-level resolution constraint unsafe.
selected_columns = [
    "advertised_start_uk",
    "advertised_start_utc",
    "advertised_start_course_local",
]

selected_population_count = temporal[selected_columns].notna().sum(axis=1)

print("\nSelected timestamp completeness pattern:")
print(selected_population_count.value_counts().sort_index().to_string())

partial_selected_rows = int(
    selected_population_count.isin([1, 2]).sum()
)

print(f"\nRows with only part of the selected timestamp set: {partial_selected_rows:,}")

Exact governance-state combinations:
temporal_resolution_status selected_branch                      decision_method decision_confidence  races
                  resolved     candidate_b course_local_dead_of_night_rejection                high  98345
                  resolved     candidate_b  stable_post_boundary_course_profile           supported  39855
                unresolved            <NA>                           unresolved          unresolved  19578
                  resolved     candidate_a course_local_dead_of_night_rejection                high  13526
                  resolved    explicit_24h          explicit_post_boundary_time     source_explicit  10352
                  resolved     candidate_a  stable_post_boundary_course_profile           supported   7387

Period/candidate structure:
       period  races  with_candidate_a_naive  with_candidate_b_naive  explicit_24h_branch
 pre_boundary 178691                  178691                  178691                    0
post_

### Design decision — one race-time record per structural race occurrence

The evidence supports a single governed temporal extension at race grain.

Its logical grain is:

> exactly one temporal record for each `core_source_race_occurrence`.

The extension must preserve:

- the existing structural race identifier and raw `date + course + off`;
- the governed IANA racecourse timezone;
- both pre-boundary candidate UK civil timestamps;
- candidate UTC and course-local timestamps where civil-time conversion is valid;
- intentional null candidate conversions at governed DST edges;
- the selected UK, UTC and course-local advertised start where resolved;
- selected branch;
- decision method;
- decision confidence;
- temporal resolution status.

The temporal extension does not alter race identity and does not overwrite raw `off`.

Because accepted Database v1 is immutable, implementation must occur through a new candidate/release boundary rather than by modifying `inside_rails_v1.sqlite3` in place.

Before physical DDL is written, the proposed cross-field constraints must be verified against all 189,043 governed temporal rows.

The next bounded question is:

> Does every governed race-time row satisfy the complete set of constraints we intend the database itself to enforce?

In [10]:
# Build explicit boolean checks for the proposed database contract. The aim is
# to prove every constraint against the complete governed temporal population
# before encoding any of them in SQLite CHECK constraints.

resolved_mask = temporal["temporal_resolution_status"].eq("resolved")
unresolved_mask = temporal["temporal_resolution_status"].eq("unresolved")

pre_mask = source_dates.lt(FORMAT_BOUNDARY)
post_mask = ~pre_mask

selected_timestamp_columns = [
    "advertised_start_uk",
    "advertised_start_utc",
    "advertised_start_course_local",
]

candidate_a_conversion_columns = [
    "candidate_a_utc",
    "candidate_a_course_local",
]

candidate_b_conversion_columns = [
    "candidate_b_utc",
    "candidate_b_course_local",
]


# Each candidate UTC/course-local pair must move together. A UTC value without
# its corresponding local representation, or vice versa, would be an invalid
# persisted state.
candidate_a_pair_consistent = (
    temporal["candidate_a_utc"].isna()
    == temporal["candidate_a_course_local"].isna()
)

candidate_b_pair_consistent = (
    temporal["candidate_b_utc"].isna()
    == temporal["candidate_b_course_local"].isna()
)


# Pre-boundary rows require both reconstructed naive candidates. Post-boundary
# rows use explicit 24-hour source time and therefore must contain no candidate
# timestamps at all.
pre_candidates_complete = (
    temporal.loc[
        pre_mask,
        ["candidate_a_uk_naive", "candidate_b_uk_naive"],
    ]
    .notna()
    .all(axis=1)
)

post_candidates_absent = (
    temporal.loc[
        post_mask,
        candidate_columns,
    ]
    .isna()
    .all(axis=1)
)


# Resolution status controls the selected canonical payload. Resolved rows must
# have all selected timestamps and a branch; unresolved rows must have neither.
resolved_selection_complete = (
    temporal.loc[resolved_mask, selected_timestamp_columns]
    .notna()
    .all(axis=1)
    & temporal.loc[resolved_mask, "selected_branch"].notna()
)

unresolved_selection_absent = (
    temporal.loc[unresolved_mask, selected_timestamp_columns]
    .isna()
    .all(axis=1)
    & temporal.loc[unresolved_mask, "selected_branch"].isna()
)


# The explicit source format occurs only after the governed boundary and must
# use the explicit branch/method/confidence combination.
explicit_state_valid = (
    temporal.loc[post_mask, "selected_branch"].eq("explicit_24h")
    & temporal.loc[post_mask, "decision_method"].eq(
        "explicit_post_boundary_time"
    )
    & temporal.loc[post_mask, "decision_confidence"].eq("source_explicit")
    & temporal.loc[post_mask, "temporal_resolution_status"].eq("resolved")
)


# Pre-boundary selected branches may only be candidate A or B. Unresolved rows
# remain null rather than receiving a guessed branch.
pre_branch_valid = temporal.loc[pre_mask, "selected_branch"].isin(
    ["candidate_a", "candidate_b"]
) | temporal.loc[pre_mask, "selected_branch"].isna()


# A selected candidate branch must itself possess valid UTC and course-local
# timestamps. This prevents selecting one of the 141 DST-invalid branches.
selected_a_valid = (
    ~temporal["selected_branch"].eq("candidate_a")
    | (
        temporal["candidate_a_utc"].notna()
        & temporal["candidate_a_course_local"].notna()
    )
)

selected_b_valid = (
    ~temporal["selected_branch"].eq("candidate_b")
    | (
        temporal["candidate_b_utc"].notna()
        & temporal["candidate_b_course_local"].notna()
    )
)


# Method and confidence are governed pairs. Encoding these relationships in the
# database will prevent individually valid labels being combined incorrectly.
method_confidence_valid = (
    (
        temporal["decision_method"].eq(
            "course_local_dead_of_night_rejection"
        )
        & temporal["decision_confidence"].eq("high")
    )
    | (
        temporal["decision_method"].eq(
            "stable_post_boundary_course_profile"
        )
        & temporal["decision_confidence"].eq("supported")
    )
    | (
        temporal["decision_method"].eq("explicit_post_boundary_time")
        & temporal["decision_confidence"].eq("source_explicit")
    )
    | (
        temporal["decision_method"].eq("unresolved")
        & temporal["decision_confidence"].eq("unresolved")
    )
)


checks = {
    "iana_timezone_complete": temporal["iana_timezone"].notna(),
    "candidate_a_conversion_pair_consistent": candidate_a_pair_consistent,
    "candidate_b_conversion_pair_consistent": candidate_b_pair_consistent,
    "pre_boundary_naive_candidates_complete": pd.Series(
        True, index=temporal.index
    ),
    "post_boundary_candidates_absent": pd.Series(
        True, index=temporal.index
    ),
    "resolved_selection_complete": pd.Series(
        True, index=temporal.index
    ),
    "unresolved_selection_absent": pd.Series(
        True, index=temporal.index
    ),
    "explicit_state_valid": pd.Series(True, index=temporal.index),
    "pre_boundary_branch_valid": pd.Series(True, index=temporal.index),
    "selected_candidate_a_convertible": selected_a_valid,
    "selected_candidate_b_convertible": selected_b_valid,
    "method_confidence_pair_valid": method_confidence_valid,
}

# Insert period-specific checks back into full-length masks so every result is
# reported as a count of violations across the complete 189,043-race population.
checks["pre_boundary_naive_candidates_complete"].loc[pre_mask] = (
    pre_candidates_complete
)
checks["post_boundary_candidates_absent"].loc[post_mask] = (
    post_candidates_absent
)
checks["resolved_selection_complete"].loc[resolved_mask] = (
    resolved_selection_complete
)
checks["unresolved_selection_absent"].loc[unresolved_mask] = (
    unresolved_selection_absent
)
checks["explicit_state_valid"].loc[post_mask] = explicit_state_valid
checks["pre_boundary_branch_valid"].loc[pre_mask] = pre_branch_valid


contract_results = pd.DataFrame(
    [
        {
            "constraint": name,
            "passed_rows": int(mask.sum()),
            "failed_rows": int((~mask).sum()),
        }
        for name, mask in checks.items()
    ]
)

print(contract_results.to_string(index=False))

print()
print(
    "Total failed constraint evaluations: "
    f"{int(contract_results['failed_rows'].sum()):,}"
)

                            constraint  passed_rows  failed_rows
                iana_timezone_complete       189043            0
candidate_a_conversion_pair_consistent       189043            0
candidate_b_conversion_pair_consistent       189043            0
pre_boundary_naive_candidates_complete       189043            0
       post_boundary_candidates_absent       189043            0
           resolved_selection_complete       189043            0
           unresolved_selection_absent       189043            0
                  explicit_state_valid       189043            0
             pre_boundary_branch_valid       189043            0
      selected_candidate_a_convertible       188905            0
      selected_candidate_b_convertible       189040            0
          method_confidence_pair_valid       189043            0

Total failed constraint evaluations: 0


### Validation correction — nullable branch values caused indeterminate Boolean results

The first complete-contract check exposed a validation-code issue rather than a temporal-data issue.

The two selected-candidate convertibility checks did not evaluate every race:

- candidate A check returned 188,905 true rows;
- candidate B check returned 189,040 true rows.

The 138 and 3 unevaluated rows correspond exactly to the previously governed DST-invalid candidate branches.

Because `selected_branch` is nullable for unresolved races, pandas propagated `pd.NA` through the Boolean implication. Those rows were therefore counted as neither passed nor failed.

The intended constraint is conditional:

> If a candidate branch is selected, that candidate must have valid UTC and course-local timestamps. If that branch is not selected — including an unresolved null selection — the constraint is satisfied.

The validation must therefore be corrected before the database contract is accepted.

In [12]:
# Re-express candidate convertibility as an explicit implication so nullable
# `selected_branch` values cannot create indeterminate pandas Boolean results.
#
# If candidate A is selected, both A conversion fields must exist.
# Otherwise the A-specific constraint is satisfied.
selected_a = temporal["selected_branch"].eq("candidate_a").fillna(False)
selected_b = temporal["selected_branch"].eq("candidate_b").fillna(False)

candidate_a_convertible = (
    temporal["candidate_a_utc"].notna()
    & temporal["candidate_a_course_local"].notna()
)
candidate_b_convertible = (
    temporal["candidate_b_utc"].notna()
    & temporal["candidate_b_course_local"].notna()
)

selected_a_valid = (~selected_a) | candidate_a_convertible
selected_b_valid = (~selected_b) | candidate_b_convertible

# Both masks must now be ordinary two-state booleans covering every governed
# race. Any remaining null would mean the validation itself is still unsafe.
assert not selected_a_valid.isna().any()
assert not selected_b_valid.isna().any()

print(
    "Selected candidate A convertibility: "
    f"{int(selected_a_valid.sum()):,} passed / "
    f"{int((~selected_a_valid).sum()):,} failed"
)

print(
    "Selected candidate B convertibility: "
    f"{int(selected_b_valid.sum()):,} passed / "
    f"{int((~selected_b_valid).sum()):,} failed"
)

print()
print(f"Candidate A selected: {int(selected_a.sum()):,}")
print(f"Candidate B selected: {int(selected_b.sum()):,}")

# Confirm specifically that none of the 141 DST-invalid candidate branches was
# selected. This is the substantive protection the future SQL constraint needs.
dst_invalid_a_selected = int(
    (
        selected_a
        & temporal["candidate_a_utc"].isna()
    ).sum()
)
dst_invalid_b_selected = int(
    (
        selected_b
        & temporal["candidate_b_utc"].isna()
    ).sum()
)

print()
print(
    "DST-invalid candidate A branches selected: "
    f"{dst_invalid_a_selected:,}"
)
print(
    "DST-invalid candidate B branches selected: "
    f"{dst_invalid_b_selected:,}"
)

Selected candidate A convertibility: 189,043 passed / 0 failed
Selected candidate B convertibility: 189,043 passed / 0 failed

Candidate A selected: 20,913
Candidate B selected: 138,200

DST-invalid candidate A branches selected: 0
DST-invalid candidate B branches selected: 0


## Evidence conclusion and authorised database-extension design

The governed Notebook 11 temporal output is suitable for database admission.

Evidence established in this notebook:

- 189,043 canonical race-time records reconcile one-to-one with all 189,043 `core_source_race_occurrence` rows;
- `date + course + off` is unique on both sides;
- 169,465 races are resolved and 19,578 remain unresolved;
- all resolved races contain complete selected UK, UTC and course-local timestamps;
- unresolved races contain no selected timestamp or branch;
- all 178,691 pre-boundary races preserve both reconstructed UK-naive candidates;
- all 10,352 post-boundary races use the explicit 24-hour source representation and contain no candidate values;
- 141 candidate conversions are intentionally withheld at governed London DST edges:
  - 96 ambiguous civil times;
  - 45 nonexistent civil times;
- no DST-invalid candidate branch is ever selected;
- candidate UTC and course-local representations are always populated or null together;
- decision method, confidence and resolution state form only the six observed governed combinations;
- every proposed row-level integrity constraint has been checked across the complete 189,043-race population.

### Database consequence

Accepted Database v1 remains immutable.

The temporal information will therefore be implemented as **Inside Rails schema Version 2**, built as a new candidate and accepted through a separate fail-closed release boundary.

Proposed generated database names:

- candidate: `inside_rails_v2_candidate.sqlite3`;
- accepted release: `inside_rails_v2.sqlite3`.

Version 2 will retain the complete Version 1 source and structural core and add one governed one-to-one race-time extension table.

### Proposed table

`core_source_race_time`

**Grain:** exactly one row for each `core_source_race_occurrence`.

The table will use `source_race_occurrence_id` as its integer primary key and foreign key to `core_source_race_occurrence`, making the relationship physically one-to-one.

It will contain:

- `source_race_occurrence_id`;
- `governance_release_id`;
- `iana_timezone`;
- `candidate_a_uk_naive`;
- `candidate_b_uk_naive`;
- `candidate_a_utc`;
- `candidate_b_utc`;
- `candidate_a_course_local`;
- `candidate_b_course_local`;
- `advertised_start_uk`;
- `advertised_start_utc`;
- `advertised_start_course_local`;
- `selected_branch`;
- `decision_method`;
- `decision_confidence`;
- `temporal_resolution_status`.

Raw `date`, `course` and `off` remain unchanged in `core_source_race_occurrence` and are not duplicated as replacement canonical fields.

### Required physical constraints

Version 2 must enforce that:

- every structural race has exactly one temporal row;
- every temporal row references exactly one structural race;
- every row has a nonblank governed IANA timezone;
- candidate A UTC and course-local values are populated or null together;
- candidate B UTC and course-local values are populated or null together;
- pre-boundary temporal states preserve both UK-naive candidates;
- explicit post-boundary states contain no candidate timestamps;
- resolved rows contain all three selected timestamps and one selected branch;
- unresolved rows contain no selected timestamps or selected branch;
- selected candidate A may be used only when its UTC and course-local conversion exists;
- selected candidate B may be used only when its UTC and course-local conversion exists;
- the explicit branch is paired only with `explicit_post_boundary_time` / `source_explicit`;
- `course_local_dead_of_night_rejection` is paired only with `high`;
- `stable_post_boundary_course_profile` is paired only with `supported`;
- `unresolved` is paired only with unresolved confidence and unresolved resolution status;
- intentional DST-edge candidate nulls remain null;
- no timestamp is invented for an unresolved or invalid civil-time candidate.

UTC database values will follow the established Inside Rails database convention of canonical UTC ISO-8601 text ending in `Z`. UK and racecourse-local values will retain their explicit civil-time offsets. UK-naive candidate values remain offset-free ISO-8601 text.

### Study-facing consequence

Version 2 will expose the selected governed race time directly alongside structural race identity.

Reader-facing studies will therefore be able to obtain:

- raw source `off`;
- canonical advertised UK timestamp;
- canonical UTC timestamp;
- canonical racecourse-local timestamp;
- resolution status and method;

without rebuilding temporal logic or joining a separate CSV.

No change to the underlying race identity is authorised.